# 第 34 课｜Pandas 标签与行情表

本 Notebook 对应「365 天量化金融」第 34 课。课程代码主线来自本地视频、课件截图和 Whisper 口播整理；在此基础上补充量化数据语义、断言、性能验证和方法边界。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/34_youtube-06Rm58cYWZg/index.html`
- 环境：`pkm-quant`
- 数据：固定随机种子生成的离线模拟行情（**非真实股票数据**）
- 状态：可运行学习稿；不代表策略收益承诺

> **数据声明**：本 Notebook 的证券名称、价格和成交量均为教学模拟数据。它用于稳定复现计算方法，不能用于评价真实证券、市场规律或策略收益。

## 本节量化问题

本实验读取 6 只模拟资产、756 个交易日的日线长表，构造“日期 × 股票”的复权收盘价宽表，再依次验证：

1. 选择一列和多列时，`Series`、`DataFrame` 的对象类型和 shape 有何不同；
2. Pandas 如何按日期标签自动对齐，以及缺失标签为何产生 `NaN`；
3. `loc` 的标签端点和 `iloc` 的位置端点如何对应；
4. `float32` 节省了多少内存、引入了多少表示误差；
5. `apply(axis=0)` 和向量化表达式如何得到相同的按股票年化收益。

输入是 `(4536, 9)` 的模拟日线长表；核心中间表 `close` 的 shape 是 `(756, 6)`，行表示日期、列表示股票。输出优先保留日期和股票标签，便于直接核对。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 先看原始行情表

先原样读取 CSV，确认字段、行数、股票数、日期数和日期 dtype。`parse_dates=["date"]` 要求 pandas 在读取时把 `date` 从文本转换为 `datetime64[ns]`。

此时数据仍是**长表**：每一行是一只股票在一个交易日的一条记录。先观察原始数据，再做任何重排。

In [2]:
# 读取并展示原始长表；最后一行保留为表格输出。
daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
print("daily.shape =", daily.shape)
print("字段 =", daily.columns.tolist())
print("股票数 =", daily["symbol"].nunique())
print("日期数 =", daily["date"].nunique())
print("date dtype =", daily["date"].dtype)

assert daily.shape == (4536, 9)
assert daily["symbol"].nunique() == 6
assert daily["date"].nunique() == 756
daily.head()

daily.shape = (4536, 9)
字段 = ['date', 'symbol', 'sector', 'open', 'high', 'low', 'close', 'adj_close', 'volume']
股票数 = 6
日期数 = 756
date dtype = datetime64[us]


,date,symbol,sector,open,high,low,close,adj_close,volume
0,2023-01-03,BANK_A,金融,11.7051,11.7522,11.6747,11.7218,11.7218,2822797
1,2023-01-04,BANK_A,金融,11.7142,11.7878,11.6199,11.6934,11.6934,1999548
2,2023-01-05,BANK_A,金融,11.6517,11.7478,11.5905,11.6864,11.6864,1974821
3,2023-01-06,BANK_A,金融,11.6692,11.7522,11.5632,11.6461,11.6461,3999555
4,2023-01-09,BANK_A,金融,11.6141,11.6557,11.4507,11.4919,11.4919,2813450


### 从长表重排为“日期 × 股票”宽表

`pivot(index="date", columns="symbol", values="adj_close")` 的三个关键参数分别决定：

- `index`：结果的行标签，这里是交易日期；
- `columns`：结果的列标签，这里是股票代码；
- `values`：每个“日期 × 股票”格子里的数值，这里是复权收盘价。

`pivot` 要求每个 `(date, symbol)` 组合唯一；若同一股票同一天有多条记录，应先明确聚合规则，而不是让重复值静默通过。`sort_index()` 默认整理行轴上的日期。

In [3]:
# 检查主键唯一后，将复权收盘价重排为带日期和股票标签的宽表。
duplicate_pairs = daily.duplicated(["date", "symbol"]).sum()
assert duplicate_pairs == 0

close = daily.pivot(
    index="date",
    columns="symbol",
    values="adj_close",
).sort_index()

assert close.index.is_unique and close.columns.is_unique
assert close.shape == (756, 6)
print("close.shape =", close.shape, "含义：日期 × 股票")
print("股票标签 =", close.columns.tolist())
close.head()

close.shape = (756, 6) 含义：日期 × 股票
股票标签 = ['BANK_A', 'ENERGY_A', 'EV_A', 'INSURE_A', 'POWER_A', 'TECH_A']


symbol,BANK_A,ENERGY_A,EV_A,INSURE_A,POWER_A,TECH_A
date,,,,,,
2023-01-03,11.7218,6.0296,27.5575,41.7118,8.0817,36.2134
2023-01-04,11.6934,6.0138,27.5189,41.9968,8.1296,36.5943
2023-01-05,11.6864,6.0281,27.8166,41.7611,8.1630,36.0263
2023-01-06,11.6461,6.1016,28.7476,42.6564,8.1751,35.9828
2023-01-09,11.4919,6.0260,28.3860,41.8431,8.1925,35.7220


## 课程示例 1：Series、DataFrame 和标签对齐

### 1.1 选择一列与选择多列

`close["BANK_A"]` 用一个列标签选择单列，结果是一维 `Series`，shape 为 `(756,)`；`close[["BANK_A", "EV_A"]]` 用列标签列表选择多列，结果仍是二维 `DataFrame`，shape 为 `(756, 2)`。

这里的方括号按**列标签**选择，不等同于 NumPy 按整数位置切片。

In [4]:
# 对比选择单列和多列后的对象类型与 shape。
bank_series = close["BANK_A"]
two_stock_frame = close[["BANK_A", "EV_A"]]

assert isinstance(bank_series, pd.Series)
assert isinstance(two_stock_frame, pd.DataFrame)
print("bank_series.shape =", bank_series.shape)
print("two_stock_frame.shape =", two_stock_frame.shape)
two_stock_frame.head()

bank_series.shape = (756,)
two_stock_frame.shape = (756, 2)


symbol,BANK_A,EV_A
date,,
2023-01-03,11.7218,27.5575
2023-01-04,11.6934,27.5189
2023-01-05,11.6864,27.8166
2023-01-06,11.6461,28.7476
2023-01-09,11.4919,28.3860


### 1.2 用确定性小例子观察自动对齐

两个 Series 做算术时，pandas 先取日期标签的**并集**，再把同一标签上的数值配对。任一侧缺少某个日期，该日期的计算结果就是 `NaN`。

下面用股票收益减基准收益计算超额收益。数值很小且固定，可以直接核对：共同日期只有 `2025-03-04` 和 `2025-03-05`。

In [5]:
# 两个收益序列的日期不完全相同，减法会按标签并集对齐。
stock_return = pd.Series(
    [0.010, -0.020, 0.030],
    index=pd.to_datetime(["2025-03-03", "2025-03-04", "2025-03-05"]),
    name="stock_return",
)
benchmark_return = pd.Series(
    [0.004, -0.001, 0.002],
    index=pd.to_datetime(["2025-03-04", "2025-03-05", "2025-03-06"]),
    name="benchmark_return",
)
aligned_excess = (stock_return - benchmark_return).rename("excess_return")

alignment_demo = pd.concat(
    [stock_return, benchmark_return, aligned_excess],
    axis=1,
)
assert aligned_excess.isna().sum() == 2
alignment_demo

,stock_return,benchmark_return,excess_return
2025-03-03,0.01,NaN,NaN
2025-03-04,-0.02,0.004,-0.024
2025-03-05,0.03,-0.001,0.031
2025-03-06,NaN,0.002,NaN


### 1.3 显式决定使用交集，而不是掩盖缺口

`Series.align(other, join="inner")` 只保留两侧共同日期。进入组合、超额收益或回归计算前，还应检查双方独有的日期。

自动对齐只保证“标签相同的数值被放在一起”，不会判断两个相同日期是否使用相同时区、交易日历、复权口径、频率或单位。也不要默认用 `fill_value=0`：只有业务上能证明“缺失就是零”时，这个假设才成立。

In [6]:
# 显式保留日期交集，并分别报告双方缺失的标签。
stock_common, benchmark_common = stock_return.align(
    benchmark_return,
    join="inner",
)
safe_excess = stock_common - benchmark_common

stock_only = stock_return.index.difference(benchmark_return.index)
benchmark_only = benchmark_return.index.difference(stock_return.index)
print("仅股票有 =", [date.date() for date in stock_only])
print("仅基准有 =", [date.date() for date in benchmark_only])

safe_alignment = pd.DataFrame(
    {
        "stock_return": stock_common,
        "benchmark_return": benchmark_common,
        "excess_return": safe_excess,
    }
)
assert safe_alignment.index.equals(
    pd.to_datetime(["2025-03-04", "2025-03-05"])
)
safe_alignment

仅股票有 = [datetime.date(2025, 3, 3)]
仅基准有 = [datetime.date(2025, 3, 6)]


,stock_return,benchmark_return,excess_return
2025-03-04,-0.02,0.004,-0.024
2025-03-05,0.03,-0.001,0.031


## 课程示例 2：`loc` 与 `iloc` 的端点

`loc` 使用日期和股票标签；标签切片包含开始、结束两个端点。`iloc` 使用整数位置，停止位置不包含，所以要对结束行位置显式 `+1`。

列也必须选择相同对象：不能让 `loc` 选择 `BANK_A、EV_A`，却让 `iloc[:, :2]` 隐式取当前表的前两列。下面先把股票标签转换为真实列位置，再验证两个结果的标签和值完全一致。

In [7]:
# 用同样的 3 个日期、2 只股票对照 loc 标签切片和 iloc 位置切片。
selected_symbols = ["BANK_A", "EV_A"]
loc_slice = close.loc[
    "2023-03-01":"2023-03-03",
    selected_symbols,
]

row_start = close.index.get_loc(pd.Timestamp("2023-03-01"))
row_stop = close.index.get_loc(pd.Timestamp("2023-03-03")) + 1
column_positions = close.columns.get_indexer(selected_symbols)
assert (column_positions >= 0).all()

iloc_slice = close.iloc[
    row_start:row_stop,
    column_positions,
]

pd.testing.assert_frame_equal(loc_slice, iloc_slice)
print("行位置 =", row_start, "到", row_stop - 1, "；iloc 停止位置 =", row_stop)
print("列位置 =", column_positions.tolist())
loc_slice

行位置 = 41 到 43 ；iloc 停止位置 = 44
列位置 = [0, 2]


symbol,BANK_A,EV_A
date,,
2023-03-01,10.9655,29.3419
2023-03-02,10.6897,28.4834
2023-03-03,10.8403,29.5063


## 课程示例 3：dtype、内存与精度

`astype("float32")` 把数值列从 64 位浮点数转换成 32 位浮点数，shape 不变。`memory_usage(index=True, deep=True)` 估计各列和索引占用的字节数；本例日期索引不会随数值 dtype 一起减半。

把 `float32` 再转回 `float64` 后与原表比较，可以测出这一次存储转换造成的最大绝对表示误差。它只描述当前价格表，不能自动证明收益复合、协方差和优化结果同样安全。

In [8]:
# 比较两种浮点 dtype 的总内存和相对原始 float64 的价格表示误差。
close64 = close.astype("float64")
close32 = close.astype("float32")

memory64 = close64.memory_usage(index=True, deep=True).sum()
memory32 = close32.memory_usage(index=True, deep=True).sum()
max_abs_error = (close64 - close32.astype("float64")).abs().to_numpy().max()

dtype_comparison = pd.DataFrame(
    {
        "memory_KiB": [memory64 / 1024, memory32 / 1024],
        "max_abs_error_vs_float64": [0.0, max_abs_error],
    },
    index=pd.Index(["float64", "float32"], name="dtype"),
)

assert close64.shape == close32.shape == (756, 6)
assert close32.dtypes.eq(np.dtype("float32")).all()
assert memory32 < memory64
dtype_comparison

,memory_KiB,max_abs_error_vs_float64
dtype,,
float64,57.507812,0.000000
float32,39.789062,0.000004


## 课程示例 4：`apply` 与向量化年化收益

### 4.1 先用确定性价格表理解 `apply(axis=0)`

`DataFrame.apply(function, axis=0)` 每次把一整列传给函数，因此这里每只股票得到一个结果。几何年化公式为：

> `(期末价格 / 期初价格) ** (每年期数 / 样本变化期数) - 1`

`**` 是幂运算。下面故意使用“每年 2 期、样本有 2 个变化区间”的小例子，因此指数等于 1：A 从 100 涨到 121，结果为 `0.21`；B 从 200 跌到 180，结果为 `-0.10`。

In [9]:
# 定义按单列价格计算的几何年化收益，并用固定小表核对 axis=0。
def annualized_return(series, periods_per_year=252):
    clean = series.dropna()
    periods = len(clean) - 1
    if periods <= 0:
        raise ValueError("至少需要两个有效价格")
    if clean.iloc[0] <= 0 or clean.iloc[-1] <= 0:
        raise ValueError("几何年化要求期初和期末价格为正")
    ending_to_start = clean.iloc[-1] / clean.iloc[0]
    return ending_to_start ** (periods_per_year / periods) - 1

demo_prices = pd.DataFrame(
    {
        "A": [100.0, 110.0, 121.0],
        "B": [200.0, 190.0, 180.0],
    },
    index=["期初", "中间", "期末"],
)
demo_annual = demo_prices.apply(
    annualized_return,
    axis=0,
    periods_per_year=2,
)

np.testing.assert_allclose(
    demo_annual.to_numpy(),
    np.array([0.21, -0.10]),
)
demo_annual.rename("annualized_return").to_frame()

,annualized_return
A,0.21
B,-0.10


### 4.2 回到完整行情：逐列 `apply` 与整表向量化

完整 `close` 没有缺失值，共有 756 个价格观测，因此相邻价格变化区间是 `756 - 1 = 755`。向量化写法先让期末和期初两行按股票标签相除，再一次对全部股票做幂运算。

本例用 `252` 个交易期代表一年，所以结果是按规则交易期频率计算的样本期几何年化收益；它不是未来预期收益。

In [10]:
# 用两种方法计算全部股票的年化收益，并按股票标签验证结果一致。
assert close.notna().all().all()
annual_apply = close.apply(
    annualized_return,
    axis=0,
    periods_per_year=252,
)

periods = len(close) - 1
ending_to_start = close.iloc[-1] / close.iloc[0]
annual_vector = ending_to_start ** (252 / periods) - 1

pd.testing.assert_index_equal(annual_apply.index, annual_vector.index)
np.testing.assert_allclose(
    annual_apply.to_numpy(),
    annual_vector.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)

annual_output = pd.DataFrame(
    {
        "apply_pct": annual_apply * 100,
        "vectorized_pct": annual_vector * 100,
        "absolute_difference": (annual_apply - annual_vector).abs(),
    }
)
print("价格观测数 =", len(close), "；变化期数 =", periods)
annual_output.round(
    {
        "apply_pct": 2,
        "vectorized_pct": 2,
        "absolute_difference": 12,
    }
)

价格观测数 = 756 ；变化期数 = 755


,apply_pct,vectorized_pct,absolute_difference
symbol,,,
BANK_A,8.63,8.63,0.0
ENERGY_A,1.53,1.53,0.0
EV_A,13.68,13.68,0.0
INSURE_A,11.53,11.53,0.0
POWER_A,3.75,3.75,0.0
TECH_A,30.54,30.54,0.0


### 4.3 正确性验证后再比较性能

`timeit.repeat(..., number=20, repeat=3)` 独立计时 3 轮，每轮执行 20 次。每轮返回的是 20 次总耗时，所以除以 `number` 后才是单次毫秒数；取最小值用于减少偶发调度干扰。

这里的 `lambda` 只把带参数的计算包装成计时器可调用的无参数函数。性能倍数取决于本机、pandas 版本、行列规模和函数复杂度，不能写成固定结论。

In [11]:
# 分开计时 apply 和向量化表达式，报告本机每次调用耗时。
def vectorized_annualized_return(frame, periods_per_year=252):
    frame_periods = len(frame) - 1
    ending_to_start = frame.iloc[-1] / frame.iloc[0]
    return ending_to_start ** (periods_per_year / frame_periods) - 1

number = 20
repeat = 3
apply_runs = timeit.repeat(
    lambda: close.apply(
        annualized_return,
        axis=0,
        periods_per_year=252,
    ),
    number=number,
    repeat=repeat,
)
vector_runs = timeit.repeat(
    lambda: vectorized_annualized_return(close),
    number=number,
    repeat=repeat,
)

apply_ms = min(apply_runs) / number * 1000
vector_ms = min(vector_runs) / number * 1000
timing = pd.Series(
    {
        "apply_ms_per_call": apply_ms,
        "vectorized_ms_per_call": vector_ms,
        "local_speedup_x": apply_ms / vector_ms,
    },
    name="本机测量值",
)
timing.to_frame()

,本机测量值
apply_ms_per_call,0.220460
vectorized_ms_per_call,0.048856
local_speedup_x,4.512430


## 安全修改：先筛选、排序并创建独立副本

`between(..., inclusive="both")` 选择包含两端的日期范围；`.eq("BANK_A")` 做逐行标签比较；两个布尔条件用 `&` 连接，不能写 Python 的 `and`。

收益依赖时间顺序，所以筛选后显式按日期排序。`.copy()` 创建独立结果，后续新增列不会修改原始 `daily`。

In [12]:
# 只保留教学所需列，按时间排序后创建独立的 3 月 BANK_A 表。
march_mask = (
    daily["date"].between(
        "2023-03-01",
        "2023-03-31",
        inclusive="both",
    )
    & daily["symbol"].eq("BANK_A")
)
march = (
    daily.loc[
        march_mask,
        ["date", "symbol", "adj_close"],
    ]
    .sort_values("date")
    .copy()
)

assert len(march) == 23
assert march["date"].is_monotonic_increasing
print("march.shape =", march.shape)
march.head()

march.shape = (23, 3)


,date,symbol,adj_close
41,2023-03-01,BANK_A,10.9655
42,2023-03-02,BANK_A,10.6897
43,2023-03-03,BANK_A,10.8403
44,2023-03-06,BANK_A,11.0317
45,2023-03-07,BANK_A,11.0235


### 在副本中新增日收益列

`pct_change(fill_method=None)` 按当前行顺序计算“本行价格 ÷ 上一行价格 − 1”，并明确禁止先用旧价格填补缺失值。第一行没有前值，因此 `daily_return` 为 `NaN`。

最后检查原始长表没有新增该列，证明这次修改只发生在 `march` 副本中。

In [13]:
# 在已排序副本中计算收益；原始 daily 保持不变。
march.loc[:, "daily_return"] = march["adj_close"].pct_change(
    fill_method=None
)

assert "daily_return" not in daily.columns
assert march["daily_return"].isna().sum() == 1
march.head()

,date,symbol,adj_close,daily_return
41,2023-03-01,BANK_A,10.9655,NaN
42,2023-03-02,BANK_A,10.6897,-0.025152
43,2023-03-03,BANK_A,10.8403,0.014088
44,2023-03-06,BANK_A,11.0317,0.017656
45,2023-03-07,BANK_A,11.0235,-0.000743


## 方法边界

- 标签对齐产生 `NaN` 是数据缺口提示，不应直接 `fillna(0)` 掩盖。
- `loc` 依赖标签语义；`iloc` 依赖当前排列位置，行列重排后整数位置会改变。
- `float32` 是否足够取决于金额尺度、累计计算和优化稳定性；本项目默认 `float64`。
- `apply` 适合没有现成向量化表达的复杂逐列/逐行逻辑；常规收益和聚合优先使用内置运算。
- 几何年化假设观测频率规则。内部缺失交易日会使“有效观测数”与真实持有时间不同，必须另行处理。
- 样本期年化收益只是历史描述统计，不是策略承诺或未来预期收益。

## 可复用结论与练习

**可复用结论**

1. 先用 pandas 完成日期和证券标签对齐，再决定是否转换为 NumPy。
2. 切片后同时核对标签、shape 和数值，不能只看行数。
3. 性能比较必须在结果一致之后进行，并报告本机测量条件。
4. 对价格做 `pct_change` 前先按证券、时间排序，并明确缺失值处理。

**练习**

- 从 `close` 中选择 `ENERGY_A` 和 `TECH_A` 的 2023 年 4 月数据，分别用 `loc` 和 `iloc` 得到同一结果，并用 `assert_frame_equal` 验证。
- 删除某只股票中间一个日期，比较自动对齐、`align(join="inner")` 和错误的 `fillna(0)` 对超额收益的影响。
- 分别用 252 个交易期和实际日历天数年化同一段收益，解释两种口径为何不同。